# Weather Dataset Loading & Preprocessing Pipeline

This notebook demonstrates a professional workflow for:

- Extracting ERA5 weather datasets
- Loading NetCDF climate files
- Merging temperature and precipitation datasets
- Selecting a single geographic location
- Dumping raw dataset

---

## Dataset Source
- ERA5 Reanalysis Dataset
- Copernicus Climate Data Store (CDS)

## Target Region
- Azamgarh, Uttar Pradesh, India


## Import Required Libraries

In [1]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import os

## Configuration

In [2]:
# Yearly folders
YEARS = ["2021", "2022", "2023", "2024", "2025"]

# Monthly folders
MONTHS = ["JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"]

# Target coordinate (Azamgarh)
TARGET_LATITUDE = 26.00
TARGET_LONGITUDE = 83.00

# Dataset directory
BASE_DIR = os.path.dirname(os.getcwd())
DATASET_DIR = Path(BASE_DIR) / "data"

## Extract ZIP Files

In [4]:
for year in YEARS:
    for month in MONTHS:

        zip_path = DATASET_DIR / "archives" / year / f"azamgarh_weather_{month}_{year}.zip"
        extract_path = DATASET_DIR / "raw" / year / month

        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(extract_path)

        print(f"{month}_{year} extracted successfully.")

JAN_2021 extracted successfully.
FEB_2021 extracted successfully.
MAR_2021 extracted successfully.
APR_2021 extracted successfully.
MAY_2021 extracted successfully.
JUN_2021 extracted successfully.
JUL_2021 extracted successfully.
AUG_2021 extracted successfully.
SEP_2021 extracted successfully.
OCT_2021 extracted successfully.
NOV_2021 extracted successfully.
DEC_2021 extracted successfully.
JAN_2022 extracted successfully.
FEB_2022 extracted successfully.
MAR_2022 extracted successfully.
APR_2022 extracted successfully.
MAY_2022 extracted successfully.
JUN_2022 extracted successfully.
JUL_2022 extracted successfully.
AUG_2022 extracted successfully.
SEP_2022 extracted successfully.
OCT_2022 extracted successfully.
NOV_2022 extracted successfully.
DEC_2022 extracted successfully.
JAN_2023 extracted successfully.
FEB_2023 extracted successfully.
MAR_2023 extracted successfully.
APR_2023 extracted successfully.
MAY_2023 extracted successfully.
JUN_2023 extracted successfully.
JUL_2023 e

## Load Instantaneous Weather Variables

This dataset contains:
- Temperature
- Pressure
- Wind
- Cloud Cover


In [3]:
instant_dfs = []

for year in YEARS:
    for month in MONTHS:

        file_path = (
            DATASET_DIR /
            "raw" /
            year /
            month /
            "data_stream-oper_stepType-instant.nc"
        )

        with xr.open_dataset(
            file_path,
            engine="netcdf4"
        ) as ds:

            df = ds.to_dataframe().reset_index()
        
        instant_dfs.append(df)

    # Combine all months
    instant_df = pd.concat(
        instant_dfs,
        ignore_index=True
    )

print(instant_df.shape)
instant_df.head()

(876480, 14)


,valid_time,latitude,longitude,t2m,d2m,sp,u10,v10,tcc,lcc,mcc,hcc,number,expver
0,2021-01-01,26.25,82.50,278.806030,277.414795,100670.617188,1.110901,-1.433517,0.0,0.0,0.0,0.0,0,0001
1,2021-01-01,26.25,82.75,279.245483,277.656982,100806.617188,1.228088,-1.414963,0.0,0.0,0.0,0.0,0,0001
2,2021-01-01,26.25,83.00,279.487671,277.762451,100763.617188,1.301331,-1.471603,0.0,0.0,0.0,0.0,0,0001
3,2021-01-01,26.25,83.25,279.891968,278.112061,100819.617188,1.397034,-1.417892,0.0,0.0,0.0,0.0,0,0001
4,2021-01-01,26.25,83.50,280.477905,278.744873,100901.617188,1.478088,-1.326096,0.0,0.0,0.0,0.0,0,0001


In [6]:
instant_df

,valid_time,latitude,longitude,t2m,d2m,sp,u10,v10,tcc,lcc,mcc,hcc,number,expver
0,2021-01-01 00:00:00,26.25,82.50,278.806030,277.414795,100670.617188,1.110901,-1.433517,0.0,0.0,0.000000,0.0,0,0001
1,2021-01-01 00:00:00,26.25,82.75,279.245483,277.656982,100806.617188,1.228088,-1.414963,0.0,0.0,0.000000,0.0,0,0001
2,2021-01-01 00:00:00,26.25,83.00,279.487671,277.762451,100763.617188,1.301331,-1.471603,0.0,0.0,0.000000,0.0,0,0001
3,2021-01-01 00:00:00,26.25,83.25,279.891968,278.112061,100819.617188,1.397034,-1.417892,0.0,0.0,0.000000,0.0,0,0001
4,2021-01-01 00:00:00,26.25,83.50,280.477905,278.744873,100901.617188,1.478088,-1.326096,0.0,0.0,0.000000,0.0,0,0001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
876475,2025-12-31 23:00:00,25.50,82.50,280.076416,279.981171,100337.054688,-1.365005,0.766006,1.0,1.0,0.010010,0.0,0,0001
876476,2025-12-31 23:00:00,25.50,82.75,280.193604,280.172577,100428.054688,-1.338638,0.562881,1.0,1.0,0.001678,0.0,0,0001
876477,2025-12-31 23:00:00,25.50,83.00,280.832275,280.787811,100449.054688,-1.270279,0.631241,1.0,1.0,0.000000,0.0,0,0001
876478,2025-12-31 23:00:00,25.50,83.25,281.400635,281.342499,100458.054688,-1.190201,0.823624,1.0,1.0,0.000000,0.0,0,0001


## Load Precipitation Dataset

In [4]:
precipitation_dfs = []

for year in YEARS:
    for month in MONTHS:

        file_path = (
            DATASET_DIR /
            "raw" /
            year /
            month /
            "data_stream-oper_stepType-accum.nc"
        )

        with xr.open_dataset(
            file_path,
            engine="netcdf4"
        ) as ds:

            df = ds.to_dataframe().reset_index()

        precipitation_dfs.append(df)

# Combine all quarters
precipitation_df = pd.concat(
    precipitation_dfs,
    ignore_index=True
)

print(precipitation_df.shape)
precipitation_df.head()

(876480, 6)


,valid_time,latitude,longitude,tp,number,expver
0,2021-01-01,26.25,82.50,0.0,0,0001
1,2021-01-01,26.25,82.75,0.0,0,0001
2,2021-01-01,26.25,83.00,0.0,0,0001
3,2021-01-01,26.25,83.25,0.0,0,0001
4,2021-01-01,26.25,83.50,0.0,0,0001


## Extract Single Location Time-Series

In [5]:
single_temp_df = instant_df[
    (instant_df['latitude'] == TARGET_LATITUDE) &
    (instant_df['longitude'] == TARGET_LONGITUDE)
].reset_index(drop=True)

single_precipitation_df = precipitation_df[
    (precipitation_df['latitude'] == TARGET_LATITUDE) &
    (precipitation_df['longitude'] == TARGET_LONGITUDE)
].reset_index(drop=True)

print(single_temp_df.shape)
print(single_precipitation_df.shape)

(43824, 14)
(43824, 6)


## Merge Temperature & Precipitation Data

In [6]:
single_temp_df["tp"] = (
    single_precipitation_df["tp"].values
)

weather_df = single_temp_df.copy()

In [7]:
weather_df.head()

,valid_time,latitude,longitude,t2m,d2m,sp,u10,v10,tcc,lcc,mcc,hcc,number,expver,tp
0,2021-01-01 00:00:00,26.0,83.0,279.765015,278.115967,100851.617188,1.140198,-1.366135,0.0,0.0,0.0,0.0,0,0001,0.0
1,2021-01-01 01:00:00,26.0,83.0,279.280396,277.819824,100915.101562,1.038345,-1.359451,0.0,0.0,0.0,0.0,0,0001,0.0
2,2021-01-01 02:00:00,26.0,83.0,280.050354,277.788391,101008.664062,1.483932,-1.602463,0.0,0.0,0.0,0.0,0,0001,0.0
3,2021-01-01 03:00:00,26.0,83.0,281.022278,277.844482,101079.031250,0.909714,-0.672821,0.0,0.0,0.0,0.0,0,0001,0.0
4,2021-01-01 04:00:00,26.0,83.0,284.075195,279.010529,101161.585938,0.838562,-0.459167,0.0,0.0,0.0,0.0,0,0001,0.0


In [ ]:
# =========================
# OUTPUT CONFIGURATION
# =========================

OUTPUT_DIR = DATASET_DIR / "processed"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_FILE = (
    OUTPUT_DIR /
    "azamgarh_weather_raw.csv"
)


# =========================
# SAVE DATASET
# =========================

weather_df.to_csv(
    OUTPUT_FILE,
    index=False
)


# =========================
# CONFIRMATION
# =========================

print("\nDataset saved successfully.")

print(f"\nLocation: {OUTPUT_FILE}")

print(f"\nTotal Rows: {weather_df.shape[0]}")

print(f"Total Columns: {weather_df.shape[1]}")


Dataset saved successfully.

Location: c:\Users\DELL\OneDrive\Desktop\weather-forecasting-system\data\processed\azamgarh_weather_raw_5years.csv

Total Rows: 43824
Total Columns: 15
